In [7]:
# ========== P1：倾向评分 ==========
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from scipy import stats
import statsmodels.formula.api as smf

df_ps = df_main.copy()
df_ps['age_75_84']  = (df_ps['age_group'] == '75-84').astype(int)
df_ps['age_85plus'] = (df_ps['age_group'] == '85+').astype(int)

ps_vars = ['gender_male', 'age_75_84', 'age_85plus',
           'baseline_k', 'drug_count_no_laxatives',
           'chf', 'ckd', 'dm',
           'diuretic', 'acei_arb', 'digoxin', 'insulin',
           'statin', 'anticoagulant']

df_ps = df_ps.dropna(subset=ps_vars + ['senna_exposed', 'y']).copy()
print(f'PSM 分析集: {len(df_ps)}')
print(f'  暴露组: {(df_ps["senna_exposed"]==1).sum()}')
print(f'  对照组: {(df_ps["senna_exposed"]==0).sum()}')

# 估计 PS
X = df_ps[ps_vars].values
y_ps = df_ps['senna_exposed'].values
lr = LogisticRegression(max_iter=2000, solver='lbfgs')
lr.fit(X, y_ps)
df_ps['ps'] = lr.predict_proba(X)[:, 1]

print('\nPS 分布:')
print(df_ps.groupby('senna_exposed')['ps'].describe())

# 共同支持区
ps_min = df_ps.groupby('senna_exposed')['ps'].min().max()
ps_max = df_ps.groupby('senna_exposed')['ps'].max().min()
print(f'\n共同支持区: [{ps_min:.4f}, {ps_max:.4f}]')

in_support = ((df_ps['ps'] >= ps_min) & (df_ps['ps'] <= ps_max)).sum()
print(f'在共同支持区内: {in_support} '
      f'({in_support/len(df_ps)*100:.1f}%)')

# 各组在共同支持区内的分布
in_supp_exp = ((df_ps['ps'] >= ps_min) & (df_ps['ps'] <= ps_max)
               & (df_ps['senna_exposed'] == 1)).sum()
in_supp_ctrl = ((df_ps['ps'] >= ps_min) & (df_ps['ps'] <= ps_max)
                & (df_ps['senna_exposed'] == 0)).sum()
print(f'  暴露组在支持区内: {in_supp_exp}')
print(f'  对照组在支持区内: {in_supp_ctrl}')

PSM 分析集: 41529
  暴露组: 24226
  对照组: 17303

PS 分布:
                 count      mean       std       min       25%       50%  \
senna_exposed                                                              
0              17303.0  0.567129  0.079608  0.274598  0.515876  0.568791   
1              24226.0  0.594918  0.082477  0.288983  0.539051  0.595033   

                    75%       max  
senna_exposed                      
0              0.620634  0.904303  
1              0.650569  0.928308  

共同支持区: [0.2890, 0.9043]
在共同支持区内: 41526 (100.0%)
  暴露组在支持区内: 24225
  对照组在支持区内: 17301


In [8]:
# ========== P2：有放回匹配 ==========
caliper = 0.02   # 初稿用 0.1，这次更严格，可试 0.01/0.02/0.05

treated = df_ps[df_ps['senna_exposed'] == 1].copy()
control = df_ps[df_ps['senna_exposed'] == 0].copy()

# KDTree 最近邻匹配（有放回）
nn = NearestNeighbors(n_neighbors=1, algorithm='kd_tree')
nn.fit(control[['ps']].values)
distances, indices = nn.kneighbors(treated[['ps']].values)
distances = distances.ravel()
indices = indices.ravel()

# 卡钳筛选
matched_mask = distances <= caliper
n_matched = matched_mask.sum()
print(f'卡钳 = {caliper}')
print(f'成功匹配的暴露组: {n_matched} / {len(treated)} '
      f'({n_matched/len(treated)*100:.1f}%)')

# 唯一被匹配的对照数（有放回可能有重复）
control_matched_idx = control.index.values[indices[matched_mask]]
n_unique_ctrl = len(set(control_matched_idx))
print(f'被匹配到的对照: {n_unique_ctrl} 个唯一 '
      f'(共使用 {len(control_matched_idx)} 次)')

# 构造匹配后数据集
treated_matched = treated.iloc[matched_mask].copy()
control_matched = control.loc[control_matched_idx].copy()
matched_df = pd.concat([treated_matched, control_matched]).reset_index(drop=True)
print(f'\n匹配后总样本: {len(matched_df)}')

卡钳 = 0.02
成功匹配的暴露组: 24225 / 24226 (100.0%)
被匹配到的对照: 9842 个唯一 (共使用 24225 次)

匹配后总样本: 48450


In [9]:
# ========== P3：SMD 平衡评估 ==========
def smd_binary(x, g):
    p0, p1 = x[g == 0].mean(), x[g == 1].mean()
    pooled = np.sqrt((p0*(1-p0) + p1*(1-p1)) / 2)
    return abs(p1 - p0) / pooled if pooled > 0 else np.nan

def smd_continuous(x, g):
    x0, x1 = x[g==0].dropna(), x[g==1].dropna()
    pooled = np.sqrt((x0.std()**2 + x1.std()**2) / 2)
    return abs(x1.mean() - x0.mean()) / pooled if pooled > 0 else np.nan

binary_set = {'gender_male', 'age_75_84', 'age_85plus', 'chf', 'ckd', 'dm',
              'diuretic', 'acei_arb', 'digoxin', 'insulin', 'statin',
              'anticoagulant'}

rows = []
for v in ps_vars:
    if v in binary_set:
        s_before = smd_binary(df_ps[v].astype(int), df_ps['senna_exposed'])
        s_after  = smd_binary(matched_df[v].astype(int),
                              matched_df['senna_exposed'])
    else:
        s_before = smd_continuous(df_ps[v], df_ps['senna_exposed'])
        s_after  = smd_continuous(matched_df[v], matched_df['senna_exposed'])
    rows.append({'Variable': v,
                 'SMD before': round(s_before, 3),
                 'SMD after':  round(s_after, 3),
                 'Improved':   '✓' if s_after < 0.1 else '✗'})

smd_table = pd.DataFrame(rows)
print(smd_table.to_string(index=False))
print(f'\n匹配后 SMD<0.1 的变量: '
      f'{(smd_table["SMD after"]<0.1).sum()}/{len(smd_table)}')
print(f'匹配后 SMD 最大: {smd_table["SMD after"].max():.3f}')
smd_table.to_excel('TableS_PSM_SMD.xlsx', index=False)

               Variable  SMD before  SMD after Improved
            gender_male       0.047      0.027        ✓
              age_75_84       0.006      0.010        ✓
             age_85plus       0.102      0.013        ✓
             baseline_k       0.174      0.001        ✓
drug_count_no_laxatives       0.203      0.007        ✓
                    chf       0.083      0.026        ✓
                    ckd       0.039      0.033        ✓
                     dm       0.017      0.020        ✓
               diuretic       0.023      0.013        ✓
               acei_arb       0.013      0.029        ✓
                digoxin       0.000      0.022        ✓
                insulin       0.009      0.012        ✓
                 statin       0.046      0.002        ✓
          anticoagulant       0.156      0.002        ✓

匹配后 SMD<0.1 的变量: 14/14
匹配后 SMD 最大: 0.033


In [10]:
# ========== P4：加权 logistic 回归 ==========
# 计算权重：暴露=1，对照=被匹配次数
weight_map = pd.Series(control_matched_idx).value_counts()
matched_df['w'] = 1.0
# 对照组权重 = 匹配次数
ctrl_positions = matched_df[matched_df['senna_exposed'] == 0].index
for pos in ctrl_positions:
    orig_hadm = matched_df.loc[pos, 'hadm_id']
    matched_df.loc[pos, 'w'] = weight_map.get(orig_hadm, 1.0)

print('权重分布:')
print(matched_df.groupby('senna_exposed')['w'].describe())

# 加权回归
formula_matched = (
    'y ~ senna_exposed + gender_male + '
    'C(age_group, Treatment("65-74")) + baseline_k + '
    'drug_count_no_laxatives + '
    'chf + ckd + dm + diuretic + acei_arb + digoxin + '
    'insulin + statin + anticoagulant'
)
model_w = smf.logit(formula_matched, data=matched_df).fit(
    disp=0, freq_weights=matched_df['w'].values
)
or_ = np.exp(model_w.params['senna_exposed'])
lo  = np.exp(model_w.conf_int().loc['senna_exposed', 0])
hi  = np.exp(model_w.conf_int().loc['senna_exposed', 1])
p   = model_w.pvalues['senna_exposed']
print(f'\nPSM 加权 senna aOR = {or_:.3f} ({lo:.3f}–{hi:.3f}), P = {p:.3g}')

权重分布:
                 count  mean  std  min  25%  50%  75%  max
senna_exposed                                             
0              24225.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
1              24225.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0

PSM 加权 senna aOR = 1.230 (1.173–1.289), P = 5.88e-18


C:\Users\13676\anaconda3\Lib\site-packages\statsmodels\base\optimizer.py:21: FutureWarning: Keyword arguments have been passed to the optimizer that have no effect. The list of allowed keyword arguments for method newton is: tol, ridge_factor. The list of unsupported keyword arguments passed include: freq_weights. After release 0.14, this will raise.
  warnings.warn(


In [11]:
# ========== P5：PSM + 聚类稳健标准误 ==========
# 用 subject_id 聚类（同一患者多次使用同一对照行）
model_cluster = smf.logit(formula_matched, data=matched_df).fit(
    disp=0,
    cov_type='cluster',
    cov_kwds={'groups': matched_df['subject_id']}
)
or_ = np.exp(model_cluster.params['senna_exposed'])
lo  = np.exp(model_cluster.conf_int().loc['senna_exposed', 0])
hi  = np.exp(model_cluster.conf_int().loc['senna_exposed', 1])
p   = model_cluster.pvalues['senna_exposed']
print(f'PSM + 聚类稳健 SE:')
print(f'  senna aOR = {or_:.3f} ({lo:.3f}–{hi:.3f}), P = {p:.3g}')

PSM + 聚类稳健 SE:
  senna aOR = 1.230 (1.141–1.325), P = 5.99e-08


In [12]:
# ========== P6：PSM 汇总表 ==========
results_psm = pd.DataFrame([
    {'Method': 'PSM (caliper=0.02, with replacement)',
     'N matched': 24225,
     'senna aOR': round(np.exp(model_w.params['senna_exposed']), 3),
     '95% CI': f"{np.exp(model_w.conf_int().loc['senna_exposed',0]):.3f}–"
               f"{np.exp(model_w.conf_int().loc['senna_exposed',1]):.3f}",
     'P': f"{model_w.pvalues['senna_exposed']:.3g}"},
    {'Method': 'PSM + cluster robust SE',
     'N matched': 24225,
     'senna aOR': round(np.exp(model_cluster.params['senna_exposed']), 3),
     '95% CI': f"{np.exp(model_cluster.conf_int().loc['senna_exposed',0]):.3f}–"
               f"{np.exp(model_cluster.conf_int().loc['senna_exposed',1]):.3f}",
     'P': f"{model_cluster.pvalues['senna_exposed']:.3g}"},
])
print(results_psm.to_string(index=False))
results_psm.to_excel('TableS_PSM_results.xlsx', index=False)
print('\n已保存 TableS_PSM_results.xlsx')

                              Method  N matched  senna aOR      95% CI        P
PSM (caliper=0.02, with replacement)      24225       1.23 1.173–1.289 5.88e-18
             PSM + cluster robust SE      24225       1.23 1.141–1.325 5.99e-08

已保存 TableS_PSM_results.xlsx


In [13]:
# ========== 汇总所有分析结果 ==========
summary = pd.DataFrame([
    {'Analysis': 'Univariable logistic',
     'senna aOR': 1.265, '95% CI': '1.205–1.328',
     'P': '2.21×10⁻²¹'},
    {'Analysis': 'Multivariable logistic (main)',
     'senna aOR': 1.190, '95% CI': '1.124–1.260',
     'P': '2.53×10⁻⁹'},
    {'Analysis': 'Multivariable (excl. drug count)',
     'senna aOR': 1.203, '95% CI': '1.137–1.272',
     'P': '1.33×10⁻¹⁰'},
    {'Analysis': 'PSM (caliper=0.02, with replacement)',
     'senna aOR': 1.230, '95% CI': '1.173–1.289',
     'P': '5.88×10⁻¹⁸'},
    {'Analysis': 'PSM + cluster robust SE',
     'senna aOR': 1.230, '95% CI': '1.141–1.325',
     'P': '5.99×10⁻⁸'},
    {'Analysis': 'Dose–response (High vs Low, total dose)',
     'senna aOR': 1.641, '95% CI': '1.495–1.802',
     'P': '2.75×10⁻²⁵'},
])
summary.to_excel('Table2_summary_all_analyses.xlsx', index=False)
print(summary.to_string(index=False))

                               Analysis  senna aOR      95% CI          P
                   Univariable logistic      1.265 1.205–1.328 2.21×10⁻²¹
          Multivariable logistic (main)      1.190 1.124–1.260  2.53×10⁻⁹
       Multivariable (excl. drug count)      1.203 1.137–1.272 1.33×10⁻¹⁰
   PSM (caliper=0.02, with replacement)      1.230 1.173–1.289 5.88×10⁻¹⁸
                PSM + cluster robust SE      1.230 1.141–1.325  5.99×10⁻⁸
Dose–response (High vs Low, total dose)      1.641 1.495–1.802 2.75×10⁻²⁵
